# Code → One-Page Explanation

**Week 4 community contribution** — Turn a single file or a few related functions into:
- A short **summary** of what the code does
- **Control flow** and **data flow** bullet points

Uses OpenRouter by default. Set `OPENROUTER_API_KEY` in `.env`.

## Imports and client

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

## Prompt and LLM call

In [2]:
SYSTEM_PROMPT = """You are a clear technical writer. Given source code, you produce a one-page explanation.

You MUST respond with valid Markdown in this exact structure:

## Summary
One short paragraph: what this code does, its main purpose, and the main idea (e.g. algorithm, pattern, or flow).

## Control flow
- Bullet points describing the order of operations: entry points, conditionals, loops, and key function calls.

## Data flow
- Bullet points describing what data goes where: inputs, outputs, main variables or structures, and how they move through the code.

Do not add any section after Data flow. Do not output anything outside this structure."""


def explain_code(code: str, model: str = "openai/gpt-4o-mini") -> str:
    """Call the LLM to get the one-page explanation."""
    if not code or not code.strip():
        return ""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Explain this code.\n\n```\n{code.strip()}\n```"},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content or ""

## Gradio UI

In [3]:
# Gradio UI is in the next cell.

## Gradio UI

In [4]:
def run_explainer(code: str, model: str) -> str:
    """Generate explanation; return markdown report."""
    if not code or not code.strip():
        return "*Paste or type some code above and click Explain.*"
    try:
        return explain_code(code, model=model or "openai/gpt-4o-mini")
    except Exception as e:
        return f"**Error:** {e}"


with gr.Blocks(title="Code Explainer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("### Paste your code below")
    code_in = gr.Textbox(
        label="Source code",
        placeholder="Paste a single file or a few related functions...",
        lines=15,
    )
    model_in = gr.Dropdown(
        choices=["openai/gpt-4o-mini", "openai/gpt-4o", "anthropic/claude-3.5-sonnet", "google/gemini-2.0-flash-001"],
        value="openai/gpt-4o-mini",
        label="Model (OpenRouter)",
    )
    btn = gr.Button("Explain")
    gr.Markdown("---")
    report_out = gr.Markdown(label="Explanation")

    btn.click(
        fn=run_explainer,
        inputs=[code_in, model_in],
        outputs=[report_out],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
